Installation of the required libraries for implementing the Retrieval-Augmented Generation (RAG) pipeline.

In [ ]:
%pip install transformers datasets langchain chromadb langchain-core langchain-community langchain-text-splitters langchain-huggingface sentence_transformers langchain-openai langchainhub

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document

# Load the IBM dataset
corpus_dataset = load_dataset("ibm-research/watsonxDocsQA", "corpus", split="train")

# Convert the dataset into LangChain Document objects
documents = []
for entry in corpus_dataset:
    # We combine document title and text for better context
    content = f"Title: {entry['title']}\n\n{entry['document']}"
    doc = Document(page_content=content, metadata={"source": entry['doc_id']})
    documents.append(doc)
print(f"Loaded {len(documents)} documents from the IBM dataset.")
# print(documents[0].page_content)
print(documents[0].metadata)



In [ ]:
# from transformers import GPT2TokenizerFast

from langchain_text_splitters import RecursiveCharacterTextSplitter

# tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)
# text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
#     tokenizer, chunk_size=100, chunk_overlap=0
# )
# docs = text_splitter.split_text(documents)
# print(docs[0])
# print("==" * 50)
# print(docs[1])
# print("==" * 50)
# print(docs[2])

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import time

# 1. Use the faster, lighter model
# sentence-transformers/all-mpnet-base-v2 is more accurate but heavier
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# 2. Process in small batches to save your laptop's RAM
batch_size = 50 
vectorstore = None

print(f"Starting embedding for {len(docs)} chunks...")

for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    
    if vectorstore is None:
        vectorstore = Chroma.from_documents(documents=batch, embedding=embeddings)
    else:
        vectorstore.add_documents(batch)
        
    print(f"✅ Completed {i + len(batch)} / {len(docs)} chunks...")
    time.sleep(0.1) # Brief pause to let CPU breathe

print("🚀 Success! Vectorstore is ready.")

In [ ]:
# 1. Define a question that should be in the IBM docs
# test_query = "How do I import a space or a project to a new deployment space?"
test_query = "How can you ensure the removal of harmful content when utilizing foundation models in the Prompt Lab?"

# 2. Search for the top 3 most relevant chunks
# 'k' is the number of documents to retrieve
retrieved_docs = vectorstore.similarity_search(test_query, k=3)


# 3. View the results
print(f"Query: {test_query}\n")
print(f"Found {len(retrieved_docs)} matching chunks.\n")

for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} ---")
    # Display the first 500 characters of the chunk
    print(doc.page_content[:500] + "...")
    # Display the source metadata we saved earlier
    print(f"Source ID: {doc.metadata.get('source')}\n")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"  # or small if laptop is slow

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.3,
        "k": 5
    }
)

In [ ]:
def format_prompt(context, question):
    return f"""
    You are an intelligent assistant for answering questions using the provided context.

    Rules:
    - Use ONLY the context
    - Give a COMPLETE and DETAILED answer
    - Do NOT answer in one word
    - If context is insufficient, say "I don't know"

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

In [ ]:
def rag_pipeline(question):
    # 1. Retrieve relevant docs
    docs = retriever.invoke(question)
    
    # 2. Combine context
    context = "\n\n".join([doc.page_content for doc in docs])
    
    
    # 3. Create prompt
    prompt = format_prompt(context, question)
    
    # 4. Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    
    # 5. Generate answer
    outputs = model.generate(**inputs, max_new_tokens=200)
    
    # 6. Decode
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer

In [ ]:
question = "How can you ensure the removal of harmful content when utilizing foundation models in the Prompt Lab?"
print(rag_pipeline(question))

In [ ]:
def debug_rag(question):
    docs = retriever.invoke(question)
    
    print("\n🔍 Retrieved Chunks:\n")
    for i, doc in enumerate(docs):
        print(f"--- Chunk {i+1} ---")
        print(doc.page_content[:300])  # preview
        print()
    
    return docs

debug_rag("How can you ensure the removal of harmful content when utilizing foundation models in the Prompt Lab?")